# 検索結果の重複を減らして多様にする (MMR)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_search/mmr_diversify.ipynb)

NLP クックブック 22-14 ／ 検索

検索結果の上位が似た文書ばかりになるのを避けたい。関連度を保ちつつ、内容の違う文書を選びたい。

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
docs = [
    "京都の紅葉の名所といえば東福寺。通天橋からの眺めが有名だ。",
    "東福寺の通天橋は紅葉の名所として知られ、秋には多くの人が訪れる。",
    "紅葉の季節、東福寺の通天橋は大変混雑する。",
    "嵐山の渡月橋周辺も、秋には山が色づいて美しい。",
    "京都の秋の味覚といえば松茸や栗を使った料理である。",
    "紅葉のライトアップは永観堂が人気で、夜間拝観が行われる。",
    "京都駅から東福寺までは JR 奈良線で一駅である。",
    "北海道の紅葉は 9 月下旬から始まり、大雪山が有名だ。",
]
doc_emb = model.encode(["passage: " + d for d in docs], normalize_embeddings=True)
query = "京都の紅葉のおすすめ"
q = model.encode("query: " + query, normalize_embeddings=True)
rel = doc_emb @ q

def mmr(rel, doc_emb, k=4, lam=0.7):
    """lam × クエリとの関連度 − (1 − lam) × 選択済みの文書との最大類似度 が最大のものを 1 つずつ選ぶ"""
    selected = [int(rel.argmax())]
    while len(selected) < k:
        rest = [i for i in range(len(rel)) if i not in selected]
        scores = [lam * rel[i] - (1 - lam) * (doc_emb[i] @ doc_emb[selected].T).max() for i in rest]
        selected.append(rest[int(np.argmax(scores))])
    return selected

print("関連度順:")
for i in np.argsort(-rel)[:4]:
    print(f"   {rel[i]:.3f} {docs[i]}")
for lam in [0.7, 0.5]:
    print(f"MMR (lambda={lam}):")
    for i in mmr(rel, doc_emb, lam=lam):
        print(f"   {rel[i]:.3f} {docs[i]}")

関連度順:
   0.890 京都の紅葉の名所といえば東福寺。通天橋からの眺めが有名だ。
   0.845 東福寺の通天橋は紅葉の名所として知られ、秋には多くの人が訪れる。
   0.844 紅葉の季節、東福寺の通天橋は大変混雑する。
   0.842 紅葉のライトアップは永観堂が人気で、夜間拝観が行われる。
MMR (lambda=0.7):
   0.890 京都の紅葉の名所といえば東福寺。通天橋からの眺めが有名だ。
   0.842 紅葉のライトアップは永観堂が人気で、夜間拝観が行われる。
   0.827 北海道の紅葉は 9 月下旬から始まり、大雪山が有名だ。
   0.813 京都の秋の味覚といえば松茸や栗を使った料理である。
MMR (lambda=0.5):
   0.890 京都の紅葉の名所といえば東福寺。通天橋からの眺めが有名だ。
   0.827 北海道の紅葉は 9 月下旬から始まり、大雪山が有名だ。
   0.813 京都の秋の味覚といえば松茸や栗を使った料理である。
   0.842 紅葉のライトアップは永観堂が人気で、夜間拝観が行われる。


In [2]:
# 選ばれた文書同士の類似度の平均 (低いほど多様)
def redundancy(ids):
    sims = doc_emb[ids] @ doc_emb[ids].T
    return float((sims.sum() - len(ids)) / (len(ids) * (len(ids) - 1)))

top = list(np.argsort(-rel)[:4])
print("関連度順:", round(redundancy(top), 3), " MMR:", round(redundancy(mmr(rel, doc_emb, lam=0.5)), 3))

関連度順: 0.908  MMR: 0.831


- MMR (Maximal Marginal Relevance) は、「クエリに近い」と「すでに選んだものと違う」の両方を満たす文書を順に選ぶ。λ が 1 なら関連度順、小さいほど多様性を重視する。
- RAG では、同じ内容のチャンクを何個も渡しても情報が増えない。MMR で多様にすると、限られた件数で広い範囲の情報を渡せる。
- 多様にしすぎると関連の薄い文書が入る。λ は 0.5〜0.8 あたりから試す。
- 完全に同じ内容の文書は、検索より前に重複除去しておく（データの収集と入出力の章の MinHash のレシピを参照）。